In [1]:
import uuid
import json
from datetime import datetime, timedelta
from typing import Optional

try:
    from tabulate import tabulate
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tabulate", "-q"])
    from tabulate import tabulate

PRICE_CATALOGUE: dict = {
    "Shirt":     80.0,
    "Pants":    100.0,
    "Saree":    200.0,
    "Suit":     350.0,
    "Jacket":   250.0,
    "Kurta":    120.0,
    "Dress":    180.0,
    "Bedsheet": 150.0,
    "Blanket":  300.0,
    "Towel":     50.0,
}

VALID_STATUSES = ["RECEIVED", "PROCESSING", "READY", "DELIVERED"]
ESTIMATED_DAYS = 2

print("✅ Configuration loaded.")
print(f"   Catalogue items : {len(PRICE_CATALOGUE)}")
print(f"   Status workflow : {' → '.join(VALID_STATUSES)}")

✅ Configuration loaded.
   Catalogue items : 10
   Status workflow : RECEIVED → PROCESSING → READY → DELIVERED


In [2]:
ORDERS: dict = {}
print("✅ In-memory database initialised.")

✅ In-memory database initialised.


In [3]:
def _now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def _validate_phone(phone):
    digits = phone.replace("+91", "").replace("-", "").replace(" ", "")
    return digits.isdigit() and len(digits) == 10

def _calculate_bill(garments):
    return round(sum(PRICE_CATALOGUE.get(g["name"], 0.0) * g["qty"] for g in garments), 2)


def create_order(customer_name, phone, garments):
    if not customer_name or not customer_name.strip():
        raise ValueError("Customer name cannot be empty.")
    if not _validate_phone(phone):
        raise ValueError(f"Invalid phone number '{phone}'. Must be 10 digits.")
    if not garments:
        raise ValueError("At least one garment is required.")
    for g in garments:
        if g["name"] not in PRICE_CATALOGUE:
            raise ValueError(f"Unknown garment '{g['name']}'. Valid: {list(PRICE_CATALOGUE.keys())}")
        if not isinstance(g["qty"], int) or g["qty"] < 1:
            raise ValueError(f"Quantity for '{g['name']}' must be a positive integer.")

    order_id   = "ORD-" + str(uuid.uuid4())[:8].upper()
    created_at = _now_str()
    est_ready  = (datetime.now() + timedelta(days=ESTIMATED_DAYS)).strftime("%Y-%m-%d")

    enriched = [
        {
            "name":       g["name"],
            "qty":        g["qty"],
            "unit_price": PRICE_CATALOGUE[g["name"]],
            "subtotal":   round(PRICE_CATALOGUE[g["name"]] * g["qty"], 2),
        }
        for g in garments
    ]

    order = {
        "order_id":        order_id,
        "customer_name":   customer_name.strip(),
        "phone":           phone.strip(),
        "garments":        enriched,
        "total_bill":      _calculate_bill(garments),
        "status":          "RECEIVED",
        "created_at":      created_at,
        "estimated_ready": est_ready,
        "status_history":  [{"status": "RECEIVED", "changed_at": created_at}],
    }

    ORDERS[order_id] = order
    return order


def update_status(order_id, new_status):
    order = ORDERS.get(order_id)
    if not order:
        raise KeyError(f"Order '{order_id}' not found.")
    new_status = new_status.upper()
    if new_status not in VALID_STATUSES:
        raise ValueError(f"Invalid status '{new_status}'. Must be one of {VALID_STATUSES}.")
    current_idx = VALID_STATUSES.index(order["status"])
    new_idx     = VALID_STATUSES.index(new_status)
    if new_idx <= current_idx:
        raise ValueError(f"Cannot move from '{order['status']}' to '{new_status}'. Forward only.")
    order["status"] = new_status
    order["status_history"].append({"status": new_status, "changed_at": _now_str()})
    return order


def list_orders(status=None, customer_name=None, phone=None, garment_type=None):
    results = list(ORDERS.values())
    if status:
        results = [o for o in results if o["status"] == status.upper()]
    if customer_name:
        results = [o for o in results if customer_name.lower() in o["customer_name"].lower()]
    if phone:
        results = [o for o in results if phone in o["phone"]]
    if garment_type:
        results = [o for o in results if any(g["name"].lower() == garment_type.lower() for g in o["garments"])]
    return results


def get_order(order_id):
    order = ORDERS.get(order_id)
    if not order:
        raise KeyError(f"Order '{order_id}' not found.")
    return order


def get_dashboard():
    orders = list(ORDERS.values())
    orders_per_status = {s: 0 for s in VALID_STATUSES}
    for o in orders:
        orders_per_status[o["status"]] += 1
    garment_counts = {}
    for o in orders:
        for g in o["garments"]:
            garment_counts[g["name"]] = garment_counts.get(g["name"], 0) + g["qty"]
    top_garments = sorted(garment_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    return {
        "total_orders":      len(orders),
        "total_revenue":     round(sum(o["total_bill"] for o in orders), 2),
        "orders_per_status": orders_per_status,
        "top_garments":      top_garments,
    }

print("✅ Business logic ready.")

✅ Business logic ready.


In [4]:
STATUS_ICONS = {"RECEIVED": "📥", "PROCESSING": "🔄", "READY": "✅", "DELIVERED": "🚚"}

def print_order(order):
    icon = STATUS_ICONS.get(order["status"], "❓")
    print("\n" + "═" * 56)
    print(f"  ORDER    : {order['order_id']}")
    print(f"  Status   : {icon} {order['status']}")
    print(f"  Customer : {order['customer_name']}  |  📞 {order['phone']}")
    print(f"  Created  : {order['created_at']}")
    print(f"  Est.Ready: {order['estimated_ready']}")
    print("─" * 56)
    rows = [[g["name"], g["qty"], f"₹{g['unit_price']:.0f}", f"₹{g['subtotal']:.0f}"] for g in order["garments"]]
    print(tabulate(rows, headers=["Garment", "Qty", "Unit ₹", "Subtotal"], tablefmt="simple"))
    print("─" * 56)
    print(f"  💰 TOTAL BILL : ₹{order['total_bill']:.2f}")
    print("═" * 56)

def print_orders_table(orders):
    if not orders:
        print("⚠️  No orders found.")
        return
    rows = [
        [o["order_id"], o["customer_name"], o["phone"],
         f"{STATUS_ICONS[o['status']]} {o['status']}",
         f"₹{o['total_bill']:.0f}", o["estimated_ready"], o["created_at"][:10]]
        for o in orders
    ]
    print(tabulate(rows, headers=["Order ID","Customer","Phone","Status","Bill","Est.Ready","Date"], tablefmt="rounded_outline"))
    print(f"\n  {len(orders)} order(s).")

def print_dashboard(data):
    print("\n" + "═" * 45)
    print("       📊  STORE DASHBOARD")
    print("═" * 45)
    print(f"  Total Orders  : {data['total_orders']}")
    print(f"  Total Revenue : ₹{data['total_revenue']:.2f}")
    print("─" * 45)
    for status, count in data["orders_per_status"].items():
        print(f"    {STATUS_ICONS[status]} {status:<12}: {count:>3}  {'█' * count}")
    print("─" * 45)
    for name, qty in data["top_garments"]:
        print(f"    👔 {name:<12}: {qty}")
    print("═" * 45)

def print_catalogue():
    print(tabulate([[k, f"₹{v:.0f}"] for k, v in PRICE_CATALOGUE.items()],
                   headers=["Garment", "Price/item"], tablefmt="rounded_outline"))

print("✅ Display helpers ready.")

✅ Display helpers ready.


In [5]:
print_catalogue()

o1 = create_order("Rahul Sharma",   "9876543210", [{"name":"Shirt","qty":3},{"name":"Pants","qty":2},{"name":"Jacket","qty":1}])
o2 = create_order("Priya Mehta",    "9123456789", [{"name":"Saree","qty":2},{"name":"Kurta","qty":3}])
o3 = create_order("Ankit Verma",    "8800001234", [{"name":"Suit","qty":1},{"name":"Shirt","qty":5},{"name":"Bedsheet","qty":2}])
o4 = create_order("Sunita Agarwal", "7012345678", [{"name":"Blanket","qty":1},{"name":"Towel","qty":4}])

for o in [o1, o2, o3, o4]:
    print_order(o)

╭───────────┬──────────────╮
│ Garment   │ Price/item   │
├───────────┼──────────────┤
│ Shirt     │ ₹80          │
│ Pants     │ ₹100         │
│ Saree     │ ₹200         │
│ Suit      │ ₹350         │
│ Jacket    │ ₹250         │
│ Kurta     │ ₹120         │
│ Dress     │ ₹180         │
│ Bedsheet  │ ₹150         │
│ Blanket   │ ₹300         │
│ Towel     │ ₹50          │
╰───────────┴──────────────╯

════════════════════════════════════════════════════════
  ORDER    : ORD-4481D7FF
  Status   : 📥 RECEIVED
  Customer : Rahul Sharma  |  📞 9876543210
  Created  : 2026-04-29 21:13:26
  Est.Ready: 2026-05-01
────────────────────────────────────────────────────────
Garment      Qty  Unit ₹    Subtotal
---------  -----  --------  ----------
Shirt          3  ₹80       ₹240
Pants          2  ₹100      ₹200
Jacket         1  ₹250      ₹250
────────────────────────────────────────────────────────
  💰 TOTAL BILL : ₹690.00
════════════════════════════════════════════════════════

══════════════

In [6]:
update_status(o1["order_id"], "PROCESSING")
update_status(o1["order_id"], "READY")
update_status(o2["order_id"], "PROCESSING")
update_status(o3["order_id"], "PROCESSING")
update_status(o3["order_id"], "READY")
update_status(o3["order_id"], "DELIVERED")

print("Status updates done.")

# Show invalid transition error
try:
    update_status(o3["order_id"], "PROCESSING")
except ValueError as e:
    print(f"❌ Expected error: {e}")

# Show audit trail
for h in get_order(o1["order_id"])["status_history"]:
    print(f"  {STATUS_ICONS[h['status']]} {h['status']} at {h['changed_at']}")

Status updates done.
❌ Expected error: Cannot move from 'DELIVERED' to 'PROCESSING'. Forward only.
  📥 RECEIVED at 2026-04-29 21:13:26
  🔄 PROCESSING at 2026-04-29 21:13:33
  ✅ READY at 2026-04-29 21:13:33


In [7]:
print("ALL ORDERS:")
print_orders_table(list_orders())

print("\nFilter → READY:")
print_orders_table(list_orders(status="READY"))

print("\nFilter → name 'mehta':")
print_orders_table(list_orders(customer_name="mehta"))

print("\nFilter → garment 'Shirt':")
print_orders_table(list_orders(garment_type="Shirt"))

ALL ORDERS:
╭──────────────┬────────────────┬────────────┬───────────────┬────────┬─────────────┬────────────╮
│ Order ID     │ Customer       │      Phone │ Status        │ Bill   │ Est.Ready   │ Date       │
├──────────────┼────────────────┼────────────┼───────────────┼────────┼─────────────┼────────────┤
│ ORD-4481D7FF │ Rahul Sharma   │ 9876543210 │ ✅ READY      │ ₹690   │ 2026-05-01  │ 2026-04-29 │
│ ORD-7D383B5A │ Priya Mehta    │ 9123456789 │ 🔄 PROCESSING │ ₹760   │ 2026-05-01  │ 2026-04-29 │
│ ORD-D377CE9F │ Ankit Verma    │ 8800001234 │ 🚚 DELIVERED  │ ₹1050  │ 2026-05-01  │ 2026-04-29 │
│ ORD-9DCE0416 │ Sunita Agarwal │ 7012345678 │ 📥 RECEIVED   │ ₹500   │ 2026-05-01  │ 2026-04-29 │
╰──────────────┴────────────────┴────────────┴───────────────┴────────┴─────────────┴────────────╯

  4 order(s).

Filter → READY:
╭──────────────┬──────────────┬────────────┬──────────┬────────┬─────────────┬────────────╮
│ Order ID     │ Customer     │      Phone │ Status   │ Bill   │ Est.Ready  

In [8]:
print_dashboard(get_dashboard())
print(json.dumps(get_dashboard(), indent=2))


═════════════════════════════════════════════
       📊  STORE DASHBOARD
═════════════════════════════════════════════
  Total Orders  : 4
  Total Revenue : ₹3000.00
─────────────────────────────────────────────
    📥 RECEIVED    :   1  █
    🔄 PROCESSING  :   1  █
    ✅ READY       :   1  █
    🚚 DELIVERED   :   1  █
─────────────────────────────────────────────
    👔 Shirt       : 8
    👔 Towel       : 4
    👔 Kurta       : 3
    👔 Pants       : 2
    👔 Saree       : 2
═════════════════════════════════════════════
{
  "total_orders": 4,
  "total_revenue": 3000.0,
  "orders_per_status": {
    "RECEIVED": 1,
    "PROCESSING": 1,
    "READY": 1,
    "DELIVERED": 1
  },
  "top_garments": [
    [
      "Shirt",
      8
    ],
    [
      "Towel",
      4
    ],
    [
      "Kurta",
      3
    ],
    [
      "Pants",
      2
    ],
    [
      "Saree",
      2
    ]
  ]
}


In [9]:
tests = [
    ("Empty name",              lambda: create_order("", "9876543210", [{"name":"Shirt","qty":1}])),
    ("Bad phone",               lambda: create_order("Test", "123", [{"name":"Shirt","qty":1}])),
    ("Unknown garment",         lambda: create_order("Test", "9876543210", [{"name":"Bikini","qty":1}])),
    ("Zero qty",                lambda: create_order("Test", "9876543210", [{"name":"Shirt","qty":0}])),
    ("Empty garments",          lambda: create_order("Test", "9876543210", [])),
    ("Backward status",         lambda: update_status(o1["order_id"], "RECEIVED")),
    ("Non-existent order",      lambda: get_order("ORD-FAKE1234")),
]

for desc, fn in tests:
    try:
        fn()
        print(f"  ❌ MISSED  : {desc}")
    except (ValueError, KeyError) as e:
        print(f"  ✅ CAUGHT  : {desc} → {e}")

  ✅ CAUGHT  : Empty name → Customer name cannot be empty.
  ✅ CAUGHT  : Bad phone → Invalid phone number '123'. Must be 10 digits.
  ✅ CAUGHT  : Unknown garment → Unknown garment 'Bikini'. Valid: ['Shirt', 'Pants', 'Saree', 'Suit', 'Jacket', 'Kurta', 'Dress', 'Bedsheet', 'Blanket', 'Towel']
  ✅ CAUGHT  : Zero qty → Quantity for 'Shirt' must be a positive integer.
  ✅ CAUGHT  : Empty garments → At least one garment is required.
  ✅ CAUGHT  : Backward status → Cannot move from 'READY' to 'RECEIVED'. Forward only.
  ✅ CAUGHT  : Non-existent order → "Order 'ORD-FAKE1234' not found."


In [11]:
def _input_garments():
    print_catalogue()
    garments = []
    while True:
        name = input("  Garment name (or 'done'): ").strip().title()
        if name.lower() == "done":
            if not garments:
                print("  Add at least one garment.")
                continue
            break
        if name not in PRICE_CATALOGUE:
            print(f"  '{name}' not in catalogue.")
            continue
        try:
            qty = int(input(f"  Qty for {name}: "))
            if qty < 1: raise ValueError()
        except ValueError:
            print("  Qty must be a positive integer.")
            continue
        garments.append({"name": name, "qty": qty})
    return garments

def interactive_menu():
    while True:
        print("\n  1.Create  2.Update Status  3.All Orders  4.Filter  5.Dashboard  6.Order Detail  0.Exit")
        choice = input("  Choose: ").strip()
        if choice == "0": break
        elif choice == "1":
            try:
                o = create_order(input("  Name: "), input("  Phone: "), _input_garments())
                print_order(o)
            except ValueError as e: print(f"  ❌ {e}")
        elif choice == "2":
            try:
                o = update_status(input("  Order ID: ").upper(), input(f"  New status {VALID_STATUSES}: "))
                print(f"  ✅ → {o['status']}")
            except (KeyError, ValueError) as e: print(f"  ❌ {e}")
        elif choice == "3": print_orders_table(list_orders())
        elif choice == "4":
            print_orders_table(list_orders(
                status=input("  Status (blank=skip): ").strip() or None,
                customer_name=input("  Name (blank=skip): ").strip() or None,
                phone=input("  Phone (blank=skip): ").strip() or None,
                garment_type=input("  Garment (blank=skip): ").strip() or None,
            ))
        elif choice == "5": print_dashboard(get_dashboard())
        elif choice == "6":
            try: print_order(get_order(input("  Order ID: ").upper()))
            except KeyError as e: print(f"  ❌ {e}")


In [14]:
interactive_menu()


  1.Create  2.Update Status  3.All Orders  4.Filter  5.Dashboard  6.Order Detail  0.Exit


  Choose:  0
